<!-- KERNEL_BANNER -->
> **Use kernel: `mrigi_tor190_v9`**  (needs `peft` for the `_COT` LoRA adapters)
>
> Set the notebook kernel to *Python (mrigi_tor190_v9)* before running.

# 23L: Backfill Q67 so all 14 models share one question set

## The problem

The 14 models in `results_23j_checkpoint.json` were **not evaluated on the same 100
questions**. They split into two groups that differ at exactly index **67**:

| Group | Models | Q67 | Result |
|---|---|---|---|
| **A** — run by 23j | the 7 non-CoT checkpoints | *Pd/ZSM-22 hydroisomerization* (original) | CUDA **OOM** at MMR → 1 `ERROR` each, so only **99** questions score |
| **B** — run by 23k | the 7 `_COT` variants | *Al organization in SSZ-13* (replacement) | works → **100** questions score |

Q67 was swapped in `zeolite_openended_100.xlsx` *between* the two runs, to remove the
OOM. Group A had already been generated against the old item.

**Why it matters.** Every base-vs-CoT comparison is slightly mismatched: the base model's
mean is over 99 questions, the CoT variant's over 100 — and the extra question is one the
base model never saw. Paired tests are unaffected (they match on index and drop the
`ERROR`), but the reported *means* are not strictly comparable.

## What this notebook does

Regenerates **only question 67** (the replacement SSZ-13 item) for the **7 group-A
models**, under both conditions, and writes the result back into the checkpoint. Nothing
else is re-run — 14 generations total, not 2,800.

Afterwards every model has the identical 100-question set and all downstream analysis is
like-for-like.

> **Run this before 30g.** 30g re-extracts and re-judges; doing it after this backfill
> means the corrected scores are already on the unified question set.

In [1]:
import os
# torch 1.12 in this env: do NOT set PYTORCH_CUDA_ALLOC_CONF=expandable_segments.
import torch, json, gc, glob
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from collections import OrderedDict

from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftConfig, PeftModel

with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            k, v = line[len('export '):].split('=', 1)
            os.environ[k] = v.strip('"').strip("'")
hf_token = os.environ['HF_TOKEN_BESTE']
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

CHECKPOINT = 'results_23j_checkpoint.json'
Q_IDX      = 67
K_MMR      = 15
MAX_NEW    = 400
TS         = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'torch {torch.__version__}   cuda={torch.cuda.is_available()}')

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


torch 2.8.0+cu128   cuda=True


In [2]:
# ---- identify which models still hold the OLD question 67 ----
gen = json.load(open(CHECKPOINT))
MODELS_ALL = [m for m in gen if 'load_error' not in gen[m]]

TARGET = pd.read_excel('zeolite_openended_100.xlsx', engine='openpyxl')
TARGET = TARGET.dropna(subset=['question', 'correct_answer', 'correct_answer_text']).reset_index(drop=True)
TARGET_Q   = TARGET.loc[Q_IDX, 'question'].strip()
TARGET_GOLD = TARGET.loc[Q_IDX, 'correct_answer_text'].strip()
print(f'canonical Q{Q_IDX} (from zeolite_openended_100.xlsx):\n  {TARGET_Q[:180]}\n')

# Only the 9 open-weight variants in Table S1 matter; the rest are not reported.
TABLE_S1 = ['Llama-3-8B-Instruct', 'base_llama_COT', 'DAPT_LR1e5', 'DAPT_LR1e5_COT',
            'synv2V2_step80', 'synv2V2_step80_COT', 'synv2_base_step80_COT',
            'fullpaper_120M_LR1e5', 'fullpaper_120M_COT']

STALE = [m for m in TABLE_S1
         if gen[m]['no_context']['detailed_results'][Q_IDX].get('query', '').strip() != TARGET_Q]
FRESH = [m for m in TABLE_S1 if m not in STALE]
OUT_OF_SCOPE = [m for m in MODELS_ALL if m not in TABLE_S1]
print(f'not in Table S1, left untouched ({len(OUT_OF_SCOPE)}): {OUT_OF_SCOPE}\n')
print(f'Table S1 models already on the canonical question ({len(FRESH)}): {FRESH}')
print(f'\nNEEDS BACKFILL ({len(STALE)}):')
for m in STALE:
    r = gen[m]['mmr']['detailed_results'][Q_IDX]
    print(f'   {m:<24} current mmr status: {r.get("model_answer", "")[:11]!r}')
print(f'\ngenerations required: {len(STALE)} models x 2 conditions = {len(STALE)*2}')

canonical Q67 (from zeolite_openended_100.xlsx):
  In Si-rich SSZ-13, why can some framework Al arrangements fail to generate exchange sites for bare divalent cations (e.g., Co2+) yet still accommodate divalent cations as hydrated 

not in Table S1, left untouched (5): ['synv2V2_final', 'synv2V2_final_COT', 'synv2_base_step80', 'synv2_base_final', 'synv2_base_final_COT']

Table S1 models already on the canonical question (5): ['base_llama_COT', 'DAPT_LR1e5_COT', 'synv2V2_step80_COT', 'synv2_base_step80_COT', 'fullpaper_120M_COT']

NEEDS BACKFILL (4):
   Llama-3-8B-Instruct      current mmr status: 'ERROR'
   DAPT_LR1e5               current mmr status: 'ERROR'
   synv2V2_step80           current mmr status: 'ERROR'
   fullpaper_120M_LR1e5     current mmr status: 'ERROR'

generations required: 4 models x 2 conditions = 8


In [3]:
# ---- model registry (adapters resolved to local snapshot dirs, as in 23k) ----
REGISTRY = {
    'Llama-3-8B-Instruct':   ('meta-llama/Meta-Llama-3-8B-Instruct', False),
    'DAPT_LR1e5':            ('aleynabeste/AllClassesAbstracts70Mmodel_LR1e5', False),
    'synv2V2_step80':        ('aleynabeste/model_LR1e5v3_synv2V2_step80', False),
    'synv2V2_final':         ('aleynabeste/model_LR1e5v3_synv2V2_final', False),
    'synv2_base_step80':     ('aleynabeste/model_LR1e5v3_synv2_base_step80', False),
    'synv2_base_final':      ('aleynabeste/model_LR1e5v3_synv2_base_final', False),
    'fullpaper_120M_LR1e5':  ('aleynabeste/model_LR1e5_fullpaper_longer_120M', False),
}
_HF = os.path.expanduser('~/.cache/huggingface/hub')


def _snapshot(repo):
    d = f"{_HF}/models--{repo.replace('/', '--')}"
    s = sorted(glob.glob(f'{d}/snapshots/*'))
    return s[-1] if s else None


_TPL = None
def _llama_template():
    global _TPL
    if _TPL is None:
        _TPL = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B-Instruct',
                                             use_fast=True, token=hf_token).chat_template
    return _TPL


def load(display):
    repo, is_adapter = REGISTRY[display]
    if is_adapter:
        d = _snapshot(repo)
        base = PeftConfig.from_pretrained(d).base_model_name_or_path
    else:
        base, d = repo, None
    tok = AutoTokenizer.from_pretrained(base, use_fast=True, trust_remote_code=True,
                                        token=hf_token, local_files_only=True)
    if getattr(tok, 'chat_template', None) is None:
        tok.chat_template = _llama_template()
    mdl = AutoModelForCausalLM.from_pretrained(base, device_map='cuda:0', torch_dtype=torch.float16,
                                               trust_remote_code=True, token=hf_token,
                                               local_files_only=True)
    if is_adapter:
        mdl = PeftModel.from_pretrained(mdl, d)
    mdl.eval()
    pipe = pipeline('text-generation', model=mdl, tokenizer=tok,
                    max_new_tokens=MAX_NEW, temperature=0.1, do_sample=True)
    return HuggingFacePipeline(pipeline=pipe), pipe, mdl, tok


def unload(llm, pipe, mdl, tok):
    try: mdl.to('cpu')
    except Exception: pass
    for o in ('model', 'tokenizer'):
        try: setattr(pipe, o, None)
        except Exception: pass
    del llm, pipe, mdl, tok
    for _ in range(3):
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()


embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
vdb = FAISS.load_local(Path('faiss_index'), embeddings, index_name='index',
                       allow_dangerous_deserialization=True)
print(f'FAISS index: {vdb.index.ntotal:,} vectors')
MMR_CONTEXT = '\n\n'.join(d.page_content for d in vdb.max_marginal_relevance_search(TARGET_Q, k=K_MMR))
print(f'MMR context for Q{Q_IDX}: {len(MMR_CONTEXT):,} chars (built once, shared by all models)')

/tmp/ipykernel_2221433/3693629855.py:62: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


FAISS index: 1,474,439 vectors
MMR context for Q67: 19,431 chars (built once, shared by all models)


In [4]:
WITH_CTX = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using the provided context together with your own knowledge. Give a clear, focused answer in 3\u20135 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Context:
{context}

Question: {question}

Answer:"""

NO_CTX = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using your own knowledge. Give a clear, focused answer in 3\u20135 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Question: {question}

Answer:"""


def strip_prompt(text):
    """Same prompt-anchored extraction 30g uses - store CLEAN answers from the start,
    so these backfilled records never carry the 23j echo bug."""
    i = text.rfind('Answer:')
    t = (text[i + 7:] if i != -1 else text).lstrip()
    if t.startswith('assistant'):
        t = t[len('assistant'):].lstrip()
    for tok in ('<|eot_id|>', '<|end_of_text|>', '<|begin_of_text|>',
                '<|start_header_id|>', '<|end_header_id|>'):
        t = t.replace(tok, '')
    return t.strip()


def generate(llm, tok, cond):
    prompt = (NO_CTX.format(question=TARGET_Q) if cond == 'no_context'
              else WITH_CTX.format(context=MMR_CONTEXT, question=TARGET_Q))
    formatted = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                        tokenize=False, add_generation_prompt=True)
    try:
        ans = strip_prompt(llm(formatted))
        return ans if ans else 'INVALID', None
    except Exception as e:
        return 'ERROR', str(e)[:300]
    finally:
        gc.collect(); torch.cuda.empty_cache()


log = []
for n, m in enumerate(STALE, 1):
    print(f'\n[{n}/{len(STALE)}] {m}')
    try:
        llm, pipe, mdl, tok = load(m)
    except Exception as e:
        print(f'   LOAD FAILED: {str(e)[:160]}')
        log.append({'model': m, 'condition': '-', 'status': 'load_error'})
        continue
    for cond in ('no_context', 'mmr'):
        ans, err = generate(llm, tok, cond)
        rec = gen[m][cond]['detailed_results'][Q_IDX]
        rec['query'] = TARGET_Q
        rec['correct_answer_text'] = TARGET_GOLD
        rec['model_answer'] = ans
        rec['full_response'] = ans
        rec['backfilled_by'] = f'23L_{TS}'
        rec.pop('error', None)
        if cond == 'mmr':
            rec['context'] = MMR_CONTEXT
        status = 'ok' if ans not in ('ERROR', 'INVALID') else ans
        print(f'   {cond:<11} {status:<8} {len(ans) if status=="ok" else 0:>5} chars'
              + (f'  {err[:80]}' if err else ''))
        log.append({'model': m, 'condition': cond, 'status': status, 'chars': len(ans)})
    unload(llm, pipe, mdl, tok)

print('\n' + pd.DataFrame(log).to_string(index=False))


[1/4] Llama-3-8B-Instruct


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/tmp/ipykernel_2221433/3693629855.py:48: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  return HuggingFacePipeline(pipeline=pipe), pipe, mdl, tok
Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/tmp/ipykernel_2221433/3735414564.py:36: LangChainDeprecationWarning: The method `Bas

   no_context  ok         492 chars
   mmr         ok         532 chars

[2/4] DAPT_LR1e5


...

   no_context  ok         407 chars
   mmr         ok         485 chars

[4/4] fullpaper_120M_LR1e5


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v9/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


   no_context  ok         377 chars
   mmr         ok         759 chars

               model  condition status  chars
 Llama-3-8B-Instruct no_context     ok    492
 Llama-3-8B-Instruct        mmr     ok    532
          DAPT_LR1e5 no_context     ok    381
          DAPT_LR1e5        mmr     ok    386
      synv2V2_step80 no_context     ok    407
      synv2V2_step80        mmr     ok    485
fullpaper_120M_LR1e5 no_context     ok    377
fullpaper_120M_LR1e5        mmr     ok    759


In [5]:
# ---- verify parity, then write back ----
import hashlib
sets = {}
for m in TABLE_S1:
    h = hashlib.md5('|'.join(r.get('query', '')
                             for r in gen[m]['no_context']['detailed_results']).encode()).hexdigest()[:8]
    sets.setdefault(h, []).append(m)

print(f'distinct question sets after backfill: {len(sets)}')
for h, ms in sets.items():
    print(f'   {h}: {len(ms)} models')

bad = [(m, c) for m in TABLE_S1 for c in ('no_context', 'mmr')
       for r in [gen[m][c]['detailed_results'][Q_IDX]]
       if r.get('model_answer') in ('ERROR', 'INVALID')]
print(f'Q{Q_IDX} still ERROR/INVALID in: {bad if bad else "none"}')

if len(sets) == 1:
    backup = f'results_23j_checkpoint_pre23L_{TS}.json'
    os.rename(CHECKPOINT, backup)
    with open(CHECKPOINT, 'w') as f:
        json.dump(gen, f, indent=2, default=str)
    print(f'\n\u2713 all {len(TABLE_S1)} Table S1 models now share one 100-question set')
    print(f'\u2713 wrote {CHECKPOINT}   (previous version saved as {backup})')
    print('\nNext: run 30g to re-extract + re-judge on the unified set.')
else:
    print('\n\u2717 question sets still differ - checkpoint NOT overwritten. '
          'Inspect the failures above before re-running.')

distinct question sets after backfill: 1
   2dfccd75: 9 models
Q67 still ERROR/INVALID in: none

✓ all 9 Table S1 models now share one 100-question set
✓ wrote results_23j_checkpoint.json   (previous version saved as results_23j_checkpoint_pre23L_20260803_192713.json)

Next: run 30g to re-extract + re-judge on the unified set.
